# 03 Performance
Baseline vs final design, profiling, memory, scalability, numerical stability, noise robustness and repeatability (report Section 6).

**Run All takes about 5–8 minutes**, mostly the 3-sensor SVR (about 1 minute per fit). Timings will differ from the report on other hardware; accuracy values match exactly.

In [ ]:
import sys
sys.path.insert(0, "../src")
from pathlib import Path
import json
import numpy as np
import pandas as pd

from degradation.data import load_dataset, split_inputs_target, TARGET
from degradation.splitting import make_test_split
from degradation.models import build_models, regression_metrics
from degradation.evaluation import build_tuned_models
from degradation import performance as P
from degradation import figures as F

RESULTS = Path("../results"); FIGURES = Path("../figures")
df = load_dataset("../data/raw/data.txt")
X, y, _ = split_inputs_target(df)
X_dev, X_test, y_dev, y_test = make_test_split(X, y)
base = build_models()
tuned = build_tuned_models(base)
THREE = ["hp_turbine_exit_pres_bar", "comp_outlet_temp_K", "comp_outlet_pres_bar"]

## 1. Baseline against final design (timing, memory, support vectors)

In [ ]:
designs = {
    "mean_baseline": (tuned["mean_baseline"], None), "linear": (tuned["linear"], None),
    "tree": (tuned["tree"], None), "svr_default": (base["svr"], None),
    "svr_tuned": (tuned["svr"], None), "svr_tuned_3sensors": (tuned["svr"], THREE),
}
rows, fitted = [], {}
for name, (model, cols) in designs.items():
    cols = cols or list(X.columns)
    r = P.time_model(model, X_dev[cols], y_dev, X_test[cols], repeats=1 if "3sensors" in name else 3)
    last = r["model"].steps[-1][1] if hasattr(r["model"], "steps") else r["model"]
    rows.append({"design": name, **regression_metrics(y_test, r["model"].predict(X_test[cols])),
                 "fit_s": r["fit_s"], "predict_us": r["predict_us_per_case"],
                 "size_kB": P.model_size_bytes(r["model"]) / 1024,
                 "n_support_vectors": len(getattr(last, "support_", [])),
                 "smo_iterations": getattr(last, "n_iter_", None) if hasattr(last, "support_") else None})
    if cols == list(X.columns):
        fitted[name] = r["model"]
design = pd.DataFrame(rows)
design.to_csv(RESULTS / "design_comparison.csv", index=False)
design.round(6)

## 2. Profiling (cProfile)

In [ ]:
print("SVR (tuned) training:")
display(P.profile_fit(tuned["svr"], X_dev, y_dev).round(3))
print("Decision tree training:")
display(P.profile_fit(tuned["tree"], X_dev, y_dev, top=5).round(3))

## 3. FLOP estimates for SVR prediction
Per support vector: 3p FLOPs for the squared distance, about 20 FLOP-equivalents for exp (an assumption), and 2 for the weighted sum.

In [ ]:
for name in ["svr_tuned", "svr_tuned_3sensors"]:
    r = design.set_index("design").loc[name]
    p = 3 if "3sensors" in name else 12
    flops = r["n_support_vectors"] * (3 * p + 22)
    print(f"{name}: {flops:,.0f} FLOPs per case, {r['predict_us']:.1f} us -> {flops / r['predict_us'] / 1e3:.2f} GFLOP/s")

## 4. Scalability

In [ ]:
exponents = {}
for name in ["linear", "tree", "svr"]:
    table, k = P.scaling_study(tuned[name], X_dev, y_dev, [1000, 2000, 4000, len(X_dev)])
    exponents[name] = k
    table.to_csv(RESULTS / f"scaling_{name}.csv", index=False)
    print(name, "fit time ~ n^%.2f" % k); display(table.round(4))

## 5. Numerical stability of least squares

In [ ]:
X14 = df.drop(columns=[TARGET, "turbine_decay", "comp_inlet_temp_K", "comp_inlet_pres_bar"])
stability = {"12 kept sensors": P.least_squares_stability(X, y),
             "14 sensors incl. duplicates": P.least_squares_stability(X14, y)}
json.dump({**stability, "scaling_exponents": exponents}, open(RESULTS / "stability_scaling.json", "w"), indent=1)
pd.DataFrame(stability)

## 6. Robustness to sensor noise

In [ ]:
noise = P.noise_robustness({k: fitted[k] for k in ["linear", "tree", "svr_tuned"]}, X_test, y_test,
                           [0, 0.0001, 0.0005, 0.001, 0.002, 0.005, 0.01])
noise.to_csv(RESULTS / "noise_robustness.csv", index=False)
F.plot_noise_robustness(noise, baseline=design.set_index("design").loc["mean_baseline", "mae"],
                        path=FIGURES / "noise_robustness.png")
noise.round(6)

## 7. Repeatability across random splits

In [ ]:
rep = P.repeatability({k: tuned[k] for k in ["linear", "tree", "svr"]}, X, y, seeds=[0, 1, 2, 3, 42])
rep.to_csv(RESULTS / "repeatability.csv", index=False)
rep.groupby("model")[["cv_mae", "test_mae"]].agg(["mean", "std", "min", "max"]).round(6)